#### Проблемы/особенности данных 

Для удобства в будущем расписала подробно проблемы/особенности. 
До того момента, пока не понадобится подробная информация о заключениях большая часть описанных ниже проблем не актуальна. 



#### Что лежит в таблицах

| Таблица         | Строк   | Одна строка                                                                                              | Главные поля                                                                                                                                                                            |
| --------------- | ------- | -------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `people`        | 11 757  | человек                                                                                                  | `id`, `sex`, `race`, `dob`, `age`; балл COMPAS (`decile_score`, `score_text`); поля анализа ProPublica — дело скрининга `c_*`, рецидив `r_*`, насильственный рецидив `vr_*`, `is_recid` |
| `compas`        | 37 578  | одна шкала одной оценки COMPAS (у оценки 3 шкалы: риск рецидива, насильственного рецидива, неявки в суд) | `compas_assessment_id`, `screening_date`, `scale_set`, `type_of_assessment`, `raw_score`, `decile_score`, `legal_status`, `custody_status`, `marital_status`                            |
| `casearrest`    | 128 183 | обвинение при аресте (на один арест — несколько строк)                                                   | `case_number`, `arrest_id`, `arrest_date`, `charge_degree`                                                                                                                              |
| `charge`        | 148 086 | обвинение по делу (в том числе по делам без ареста — штрафы, повестки)                                   | `case_number`, `charge_number`, `offense_date`, `date_charge_filed`, `charge_degree`, `charge` (текст), `statute`; `filing_date` — заглушка 1800-01-01                                  |
| `jailhistory`   | 22 111  | пребывание в окружной тюрьме (jail): арест, ожидание суда, короткие сроки                                | `in_custody`, `out_custody` (дата + время без AM/PM), `dob`                                                                                                                             |
| `prisonhistory` | 4 945   | срок в тюрьме штата (prison) по приговору                                                                | `in_custody`, `out_custody` (только дата; «ещё сидит» = 2020-01-01)                                                                                                                     |


**I. Время и даты**

1) Отсутствие точного времени 
	**jail:** Время в без AM/PM в 12-часовом формате
	**charge и casearrest** указана только дата, время везде 00.00
	
	при построении петри прошлой версии собственно и возникал вопрос, как упорядочивать состояния арест/обвинение/заключение (в итоге, чтобы не гадать, эти состояния просто склеивались).  
	
	
2) **Charge **, 
	- (`date_charge_filed` — точная копия 'offense_date', `filing_date` — везде заглушка ). Точно не известно, данная информация о времени, когда было совершено преступление, или когда вынесено обвинение.
	
	 Далее использовала список обвинений, как признак, в связи с чем появляется вопрос - использовать ли в момент X информацию о уже имеющихся обвинениях, если по ним еще не было арестов и/или заключений (по логике - это подсказка о возможном заключении в будущем)

3) **Заглушки и особые значения дат** (проверены все поля с датами во всех таблицах `compas.db`)

| Таблица.поле                                                | Значение                           | Записей   | Доля          | Что значит                                 | Как обрабатывается                                                                              |
| ----------------------------------------------------------- | ---------------------------------- | --------- | ------------- | ------------------------------------------ | ----------------------------------------------------------------------------------------------- |
| `prisonhistory.out_custody`                                 | **2020-01-01**                     | 679       | 13,7 %        | «ещё сидит» на момент выгрузки             | выход = конец наблюдения 01.04.2016, `ongoing` (Б4)                                             |
| `jailhistory.out_custody`                                   | **2020-01-01**                     | 8         | 0,04 %        | «ещё сидит»                                | так же (Б4)                                                                                     |
| `people.c_jail_in`, `c_jail_out`, `days_b_screening_arrest` | `NULL` / пустая строка `''` | 742 / 438 | 6,3 % / 3,7 % | два вида пропуска с **разным смыслом** (см. ниже) | поля ProPublica в графе не используются; люди не выбрасываются (Б6) |
| `casearrest.arrest_date`                                    | 2017-04-16, 2020-09-19, 2049-11-10 | 3         | 0,002 %       | даты после конца выгрузки — ошибки ввода   | удаляются (А5)                                                                                  |




4) **Дубли: что удаляется** (`build_graph`, шаг 1.2; дубль = совпадение всех столбцов, кроме служебного `id`)

| Таблица         | Какие столбцы сравниваются                                                                      | Строк   | Дублей | Удалено | Почему                                                                             |
| --------------- | ----------------------------------------------------------------------------------------------- | ------- | ------ | ------- | ---------------------------------------------------------------------------------- |
| `charge`        | все, кроме `id` (дело, номер обвинения, даты, степень, текст, статья, агентство, `person_id` …) | 148 086 | 13 пар | **13**  | настоящие повторы: совпадает даже `charge_number` — одно обвинение записано дважды |

- **`charge`, пример настоящего дубля:** человек 5579, дело 13004734CF10A, обвинение № 7 «Poss Wep Conv Felon» (F2) — строки id 104514 и 104515 совпадают во всех полях. Ещё пример — дело 14006415MU10A (человек 5263): все 4 обвинения записаны дважды.

**II. Особенности данных до 2013**

- jail только с 01.01.2013
- prison, аресты и обвинения — с 1960–1970-х

| Таблица                                | Период в базе           | Записей всего | До 2013 | Доля до 2013     |
| -------------------------------------- | ----------------------- | ------------- | ------- | ---------------- |
| `jailhistory`                          | 01.01.2013 — 31.03.2016 | 22 111        | 0       | **0 %**          |
| `prisonhistory` (вход)                 | 1974 — 2016             | 4 945         | 3 775   | **==🔴76,3 %==** |
| `casearrest` (аресты «человек + дата») | 1975 — 2016             | 55 735        | 31 534  | 56,6 %           |
| `charge` (`offense_date`)              | 1961 — 2016             | 148 086       | 89 407  | 60,4 %           |
| `people` (скрининг COMPAS)             | 01.01.2013 — 31.12.2014 | 11 757        | 0       | 0 %              |
 Jail по годам входа: 2013 — 9 119, 2014 — 9 143, 2015 — 3 261, 2016 — 588 (после 2014 новых людей в выборке нет, видны только их повторные заключения).


| Срок prison                                            | Сроков | Доля       | «Ещё сидит» (выход 2020-01-01) | Людей | Как используется в графе                                                                                 |
| ------------------------------------------------------ | ------ | ---------- | ------------------------------ | ----- | -------------------------------------------------------------------------------------------------------- |
| целиком до 2013 (выход до 01.01.2013)                  | 3 628  | **73,4 %** | 0                              | —     | только признаки истории                                                                                  |
| идёт на 01.01.2013 (вход до, выход после)              | 147    | 3,0 %      | 5 из 147 (3 %)                 | —     | начальные: не узлы графа, трасса начинается с выхода; 5 человек без выхода до 01.04.2016 — трассы убраны |
| вход с 01.01.2013, сидит на момент окончания наблюдени | 1 170  | **23,7 %** | ==🔴674 из 1 170 (58 %)==      | 1 110 | узлы графа                                                                                               |
| **всего**                                              | 4 945  | 100 %      | 679 (14 %)                     | 2 068 |                                                                                                          |

- С 2013 почти каждый арест сопровождается заключением в jail (~90%), до 2013 
- prison с входом с 2013: в 73,8 % случаев ему предшествует jail, закончившийся в пределах ±1 дня (перевод jail → prison). 
- Арест в течение года до входа в prison: до 2013 — 65 %, с 2013 — 75 % (медиана от ареста до входа 103 и 150 дней).


III. Привязка арестов к заключениям 

Информация в таблицах с заключениями:
in_custody`, `out_custody, фио, id.

Информация в таблице с арестами:
id, **case_number**`, `arrest_id`, `arrest_date`, `charge_degree 

Информация в таблице с обвинениями:
id, **`case_number`**, `charge_number`, `offense_date`, `date_charge_filed`, `charge_degree`, ==🟡`charge` (текст)==, `statute`

Проблема поиска информации о заключении - напрямую связываются признаки из таблицы people по id, c арестами и обвинениями ситуация сложнее. 

Самый очевидный способ - по дате заключения + id человека связывается арест.
Потом по **case_number** из таблицы с обвинениями (charge) вытаскивается текст обвинения, парсится и делится на 15 типов.

| Заключение           | Входов | Арест в тот же день       | Арест только накануне | Арест только на следующий день | Нет ареста в [−1; +1] день |
| -------------------- | ------ | ------------------------- | --------------------- | ------------------------------ | -------------------------- |
| jail (только с 2013) | 22 111 | ==🟢18 347 (**83,0 %**)== | 2 033 (**9,2 %**)     | 98 (0,4 %)                     | 1 633 (7,4 %)              |
| prison до 2013       | 3 775  | 9 (0,2 %)                 | 3 (0,1 %)             | 1 (0,0 %)                      | 3 762 (**99,7 %**)         |
| prison с 2013        | 1 170  | 1 (0,1 %)                 | 1 (0,1 %)             | 0                              | 1 168 (**99,8 %**)         |
| prison всего         | 4 945  | 10 (0,2 %)                | 4 (0,1 %)             | 1 (0,0 %)                      | ==🔴4 930 (**99,7 %**)==   |
т.е. при проверке в отрезке [N-1;N] подвязывается информация о аресте jail в большей части случаев.

предложение - для остальных случаев использовать Unknown 


**prison с 2013: переход из jail** (`1911/данные/custody_linked.csv` — 1 109 сроков prison с входом с 2013 после шагов 1.2–1.4 из 1 170 в базе; правило 4.2: перевод = jail закончился меньше чем за сутки до / после входа в prison и между входом в jail и входом в prison нет новых арестов и обвинений)

| Что перед входом в prison                                                                                    | Сроков    | Доля       |
| ------------------------------------------------------------------------------------------------------------ | --------- | ---------- |
| **jail закончился в тот же календарный день** (разрыв < 1 сут.)                                              | **849**   | **76,6 %** |
| ↳ без новых арестов и обвинений → **перевод jail → prison** (склеивается с jail в один узел, наследует дело) | 668       | 60,2 %     |
| ↳ с новыми арестами / обвинениями → не перевод                                                               | ==🟡181== | 16,3 %     |
| &nbsp;&nbsp;&nbsp;— только новые аресты                                                                      | 146       | 13,2 %     |
| &nbsp;&nbsp;&nbsp;— новые аресты и обвинения                                                                 | 34        | 3,1 %      |
| &nbsp;&nbsp;&nbsp;— только новые обвинения                                                                   | 1         | 0,1 %      |
| jail ещё идёт (выход из jail позже входа в prison больше чем на сутки)                                       | 16        | 1,4 %      |
| ближайший jail закончился за 1–7 дней                                                                        | 1         | 0,1 %      |
| за 8–30 дней                                                                                                 | 3         | 0,3 %      |
| за 31–365 дней                                                                                               | 88        | 7,9 %      |
| больше года назад                                                                                            | 110       | 9,9 %      |
| jail до входа в prison нет совсем                                                                            | 42        | 3,8 %      |
| **всего**                                                                                                    | **1 109** | 100 %      |

==🟡«новые» аресты== 

| Что за арест                                          | Арестов | Доля | Смысл                                                                                                           |
| ----------------------------------------------------- | ------- | ---- | --------------------------------------------------------------------------------------------------------------- |
| по старому делу (дело уже было раньше)                | 159     | 56 % | повторное оформление по открытым делам (доставка в суд, приговор, ордер); из них 42 — по делу самого этого jail |
| по новому делу, правонарушение **до** входа в jail    | 77      | 27 % | старые эпизоды, по которым обвинение предъявили, пока человек сидит                                             |
| по новому делу, правонарушение в день входа в jail    | 6       | 2 %  | тот же эпизод, оформленный отдельным делом                                                                      |
| по новому делу, правонарушение **после** входа в jail | 42      | 15 % | правонарушение в заключении или ошибка даты                                                                     |

Примеры:
- **Человек 3391**: jail 24.03.2013 → prison 24.10.2013. 26.04.2013 — повторные аресты по старым делам 2010–2012 (кража авто, наркотики) и новое дело 13006154CF10A с датой 26.04.2013: Felony Petit Theft, Unlaw Use False Name/Identity, Resist/Obstruct W/O Violence.
- **Человек 4306**: jail 20.01.2013 (Robbery W/Firearm, F1) → prison 05.12.2013. Новое дело 13014396CF10A — **Battery Upon Detainee** (F3), дата 08.10.2013, арест 13.10.2013: правонарушение в самом jail.
- **Человек 3328**: jail 11.04.2014 (кражи со взломом) → prison 28.08.2014. За время jail — аресты по 4 делам: 16.04 и 17.04 — новые дела по кражам 11.04 и 21.03 (до входа), 18.04 — старое дело 2013 года, 13.05 — дело 14006694CF10A (Burglary Unoccupied Dwelling, Grand Theft, Criminal Mischief, дата 12.04 — день после входа, вероятно, тот же эпизод).
- **Человек 7766**: восемь коротких jail в 2013 (наркотики, trespass); последний jail 16.11.2013 → prison 02.01.2014. Аресты 19.12.2013 — по старым делам весны 2013 (Possession of Cocaine, Possession of Oxycodone), новых обвинений нет: похоже на приговор сразу по нескольким делам перед prison.


Вывод - считать переводом jail закончился в тот же календарный день** (разрыв < 1 сут.). Остальное - Unknown.



**IV. OldCase — повторный арест по старому делу**


**Правило** (`build_graph`, 4.1): к заключению берутся аресты в [вход − 1 день; вход] и их дела. Дело **старое**, если по нему у человека уже было заключение в окне с 01.01.2013.
- только старые дела → причина `rearrest_old_case`, **`OldCase`**;
- новое + старое дело → `new_offense`, состояние — типы нового дела, старое записывается в поля `rearrest_case_ids`, `rearrest_first_arrest_date`, `original_case_degrees`, `rearrest_prev_custody_ids`, `rearrest_new_charges_n`.

Сколько** (`1911/данные/custody_linked.csv`, 23 177 заключений)

|                                                                     | Заключений                   | Доля                                                |
| ------------------------------------------------------------------- | ---------------------------- | --------------------------------------------------- |
| со старым делом всего                                               | 4 963                        | ==🔴21,4 %==                                        |
| ↳ только старое дело → **`OldCase`**                                | 3 470 (jail 3 469, prison 1) | 15,0 % (в графе — 15,4 % эпизодов: 3 470 из 22 507) |
| ↳ новое + старое → `new_offense` с пометкой                         | 1 493                        | 6,4 %                                               |
| новые обвинения по старому делу (с датой после прошлого заключения) | 12                           | —                                                   |
| новые дела                                                          | 15 406                       | 66,5 %                                              |

- **Скрытые старые дела (левое усечение).** «Старое» определяется по заключению в окне, а не по первому аресту. 757 заключений `new_offense`, у которых **все** дела впервые арестованы больше чем за день до входа, считаются новыми; у 599 из них дело открыто **до 2013** — заключение по нему просто не видно (jail только с 2013).


**V. Не привязавшееся**

«Привязано» — обвинение попало хотя бы в одно заключение графа (`1911/данные/custody_charges.csv`: правило 4.1 — дело ареста в [вход − 1; вход], обвинения с датой не позже дня входа; правило 5 — наследование при переводе jail → prison). 

|                                          | Обвинений | Привязано           | Не привязано           |
| ---------------------------------------- | --------- | ------------------- | ---------------------- |
| вся таблица `charge`                     | 148 073   | 43 911 (**29,7 %**) | ==🟡104 162 (70,3 %)== |
| дата правонарушения до 2013              | 89 407    | 5 294 (5,9 %)       | 84 113 (94,1 %)        |
| **окно графа** (01.01.2013 – 31.03.2016) | 58 666    | 38 617 (**65,8 %**) | 20 049 (34,2 %)        |


Основная мысль - есть много информации, которая не привязывается к заключениям, но ее можно использовать.
Проблема - избыточность информации у одних и недостаток у других.

**VI. Правое усечение: как выбран конец наблюдения (01.04.2016)**

**1) Где кончаются таблицы**

| Таблица | Последняя дата (без заглушек и ошибок) |
| --- | --- |
| `casearrest.arrest_date` | **30.03.2016** (без 2017 / 2020 / 2049) |
| `charge.offense_date` | **30.03.2016** |
| `jailhistory.in_custody` | **31.03.2016** |
| `jailhistory.out_custody` | 19.04.2016 (без 2020-01-01) |
| `prisonhistory.in_custody` | 15.04.2016 |
| `prisonhistory.out_custody` | 16.04.2016 (без 2020-01-01) |

Выгрузка сделана в апреле 2016: аресты, обвинения и входы в jail обрываются в конце марта, а выходы и prison тянутся ещё на 2–3 недели.

** Что сделано** (`build_graph`, правка 6; `cleaning_report.csv`)
- Конец наблюдения — **01.04.2016 во всех таблицах и на всех этапах** (граф, этап 2: эпизоды на свободе, цели).
- Заключения с входом не раньше 01.04.2016 убираются: **25** (это prison).
- Выход после 01.04.2016 → «ещё сидит» (`ongoing`, выход = конец наблюдения): **85** заключений (в исходной базе реальный выход после 01.04.2016 — у 62 jail и 26 prison; часть из них убрана другими правилами).
- Последний шаг каждого человека цензурирован на 01.04.2016.




